In [4]:
import os
import json
import chromadb

from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown
from chromadb.config import Settings

load_dotenv(override=True)



True

In [6]:
chroma_client = chromadb.PersistentClient(path="./")

collection = chroma_client.get_or_create_collection(name="restaurant_vector_db")

In [ ]:
# llm_client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=os.getenv("OPEN_ROUTER_KEY"),
# )

llm_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
)

# MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free, NVIDIA)
MODEL = "openai/gpt-oss-120b"
EMBEDDING_MODEL = "nvidia/nemotron-3.5-embed-8b:free"

In [ ]:
def create_embeddings(texts):
    r = llm_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)    # one request for the whole list
    return [item.embedding for item in r.data]     

## Load + chunk

The same code as Day 30. The documents are in `chai_wala_docs`.

In [ ]:
from pathlib import Path

def load_documents(folder):
    docs = []
    for path in sorted(Path(folder).glob("*.txt")):              # every .txt file in the folder
        docs.append({"source": path.name, "text": path.read_text(encoding="utf-8")})
    return docs

def chunk_by_paragraph(text):
    parts = [p.strip() for p in text.split("\n\n")]              # a blank line = a new paragraph
    return [p for p in parts if len(p) > 30]                      # skip very short parts, like titles

def chunk_documents(docs):
    chunks = []
    for doc in docs:
        for i, text in enumerate(chunk_by_paragraph(doc["text"])):
            chunks.append({"id": f"{doc['source']}-{i}", "source": doc["source"], "text": text})
    return chunks

chunks = chunk_documents(load_documents("chai_wala_docs"))
print(len(chunks), "chunks")

In [ ]:
menu = [
    {"name": "Zinger Burger", "price": 850},
    {"name": "Chicken Biryani", "price": 450},
    {"name": "Club Sandwich", "price": 700}
]

complaint_list = []

order_list = [
    "OR-1045",
    "OR-1046",
    "OR-1047"
]

In [ ]:
def fetch_menu():
    return menu

def log_complaint(customer_name, order_id, phone, complaint):
    temp_dict = {
        "customer_name": customer_name,
        "order_id": order_id,
        "phone": phone,
        "complaint": complaint,
        "status": "Pending"
    }
    complaint_list.append(temp_dict)

    return {
        "message": f"Saved! Total Complaint: {len(complaint_list)}"
    }

def validate_order_id(order_id):
    if order_id in order_list:
        return True
    return False


tools = [
    {
        "type": "function",
        "function": {
                "name": "fetch_menu",
                "description": "Get the restaurant menu",
                # no inputs
                "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "validate_order_id",
            "description": "Validate order id before calling log_complaint tool it return True/False",
            "parameters": {
                "type": "object", 
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                },
                "required": ["order_id"],
                "additionalProperties": False,

            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "log_complaint",
            "description": "Log the complain",
            "strict": True,
            "parameters": {
                "type": "object", 
                "properties": {
                    "customer_name": {
                        "type": "string",
                        "description": "Customer name who is logging complain.",
                    },
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                    "phone": {
                        "type": "string",
                        "description": "Customer phone number or '-' ",
                    },
                    "complaint": {
                        "type": "string",
                        "description": "customer complain should me not more than 8 words",
                    }
                },
                "required": ["customer_name", "order_id", "phone", "complaint"],
                "additionalProperties": False,

            },
        }
    }

]

In [ ]:
messages = [
    {
      "role": "system",  # User | Assistant | System
      "content": """
       You are the AI Customer Support Assistant for **Xander Restaurant**. Your job is to answer customer queries in a friendly, professional, and helpful way.

        ### Tools
        1. **`fetch_menu`** — Call whenever customers ask about menu items, prices, ingredients, or food recommendations. Use the tool's response to provide accurate information.
        2. **`log_complaint`** — Call whenever a customer raises a complaint. Collect the necessary details and log the complaint using this schema:

        ```python
        complaint = {
            "customer_name": "",
            "phone": "",
            "complaint": "",
            "order_id": "",
            "status": "Pending"
        }
        ```

        3. **`validate_order_id`** — Before logging complaint trigger this tool call validate_order_id to validate wether the order id is valid or not then call log_complaint


        Ask for missing necessary details before logging. The `order_id` can be empty if unavailable. Set the initial status to `"Pending"`. Confirm the complaint is logged only after the tool succeeds.

        ### Guidelines
        - Keep replies short, natural, and friendly.
        - Never invent menu details, prices, or restaurant policies.
        - If you don't know something, politely say so.
        - Make every customer feel valued.
        - All prices are in Pakistani Rupees. Write them as "Rs 850".
      """
    }
]


In [ ]:
function_names = {
    "fetch_menu": fetch_menu,
    "log_complaint": log_complaint,
    "validate_order_id": validate_order_id
}

def ask_assistant(question):

    messages.append({
        "role": "user", 
        "content": question
    })

    while True:
        completion = llm_client.chat.completions.create(
                model=MODEL, 
                messages=messages, 
                tools=tools,
                temperature=0
            )
           
        reply = completion.choices[0].message
        messages.append(reply)
    
        # reply_json = reply.model_dump(exclude_none=True)
        # reply_json.pop("reasoning", None)            # hide the model's long thinking
        # reply_json.pop("reasoning_details", None)
        # print("LLM REPLY:", json.dumps(reply_json, indent=2))
    
        if not reply.tool_calls: 
            return Markdown(reply.content)
            break                        # model asked for a tool
       
        for call in reply.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = function_names[call.function.name](**args)     # run our Python function
            print("Tool called:", call.function.name, args, result)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})        

In [ ]:
ask_assistant("i ordered burger but roll was served WTH!!! and the bill was made with burger, my name is Sam, OR-1045, phone is 0322342423")

In [ ]:
print(json.dumps(complaint_list))



Create a new tool validate_order to validate orderId when logging complaint
if order id does not gets validated say user please provider correct Order ID